# Climate Embedding Variables (CEV) — PatchTST & Informer

> **참고용 코드**: sample_data/ 에는 참고용 샘플 1개씩만 포함되어 있으며, 실제 실행 시에는 전체 WorldClim 월별 자료(1990-2024) 및 dem.asc를 별도로 받아 아래 경로에 위치시켜야 함.

In [ ]:
# === Cell 1. Config + 리소스 자동 감지 ================================
import os, re, glob, math, time, json, sys
import numpy as np
import torch, torch.nn as nn
from concurrent.futures import ThreadPoolExecutor
from sklearn.decomposition import PCA
import rasterio

SEED = 42
MODELS = "patchtst,informer,rawpca"
TRAIN_SUBSAMPLE = "all"
SAVE_REPS, LOAD_WEIGHTS, RESUME = False, False, False

BASE = "WorldClim_10min_1990-2024"
DIRS = {"tmin": os.path.join(BASE, "wc2.1_cruts4.09_10m_tmin_1990-2024"),
        "tmax": os.path.join(BASE, "wc2.1_cruts4.09_10m_tmax_1990-2024"),
        "prec": os.path.join(BASE, "wc2.1_cruts4.09_10m_prec_1990-2024")}
PREFIX = {"tmin": "wc2.1_cruts4.09_10m_tmin", "tmax": "wc2.1_cruts4.09_10m_tmax", "prec": "wc2.1_cruts4.09_10m_prec"}
DEM_PATH = "dem.asc"
OUTDIR = os.path.join("CEV_output", f"seed{SEED}")
os.makedirs(OUTDIR, exist_ok=True)
for _p in (DEM_PATH,) + tuple(DIRS.values()):
    assert os.path.exists(_p), f"경로 없음: {_p}"
CHANNELS = ["tmin", "tmax", "prec"]; N_MONTHS = 420

LOG = open(os.path.join(OUTDIR, "log.txt"), "a", encoding="utf-8")
def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); LOG.write(s + "\n"); LOG.flush()

VAR_THRESH = 0.95
VAR_THRESH_REPORT = (0.90, 0.95, 0.99)
MAX_K = 200
FIXED_K_SENS = 19
D_MODEL, N_HEADS, E_LAYERS, D_FF, DROPOUT = 128, 8, 3, 256, 0.2
PATCH_LEN, STRIDE = 12, 12
INFORMER_FACTOR = 5
MASK_RATIO, MASK_SPAN = 0.40, 12
EPOCHS = 100
LR = 5e-4
VAL_SUBSAMPLE = 10_000
VAL_SEED = 1234
RAW_FIT_N = 100_000

N_CPU = os.cpu_count() or 8
torch.set_num_threads(N_CPU)
IO_WORKERS = min(12, N_CPU)
if torch.cuda.is_available():
    DEVICE = "cuda"; _p = torch.cuda.get_device_properties(0); VRAM_GB = _p.total_memory / 1e9
    USE_AMP = True; BATCH = 512; GPU_NAME = _p.name
else:
    DEVICE, VRAM_GB, USE_AMP, BATCH, GPU_NAME = "cpu", 0.0, False, 256, "-"
INFER_BATCH = BATCH * 2
torch.manual_seed(SEED); np.random.seed(SEED)
log(f"=== seed={SEED} models={MODELS} epochs={EPOCHS} train_px={TRAIN_SUBSAMPLE} ===")
log(f"DEVICE={DEVICE} GPU={GPU_NAME} VRAM={VRAM_GB:.1f}GB AMP={USE_AMP} BATCH={BATCH} CPU={N_CPU}")
log(f"원자료: {BASE}\n출력: {OUTDIR}")

In [ ]:
# === Cell 2. 전지구 land 로딩 (dem 마스크 + 병렬 읽기) ================
def sorted_month_files(dirpath, prefix):
    files = glob.glob(os.path.join(dirpath, prefix + "_*.tif"))
    files = sorted(files, key=lambda f: tuple(map(int, re.search(r"_(\d{4})-(\d{2})\.tif$", f).groups())))
    assert len(files) == N_MONTHS, f"{prefix}: {len(files)}"
    return files
file_lists = {c: sorted_month_files(DIRS[c], PREFIX[c]) for c in CHANNELS}
with rasterio.open(DEM_PATH) as dem:
    G_H, G_W = dem.height, dem.width; G_TRANS = dem.transform
    dem_arr = dem.read(1); DEM_NODATA = dem.nodata if dem.nodata is not None else -9999.0
land = np.isfinite(dem_arr) & (dem_arr != DEM_NODATA)
LAND_ROWS, LAND_COLS = np.where(land); n_land = LAND_ROWS.size
X = np.empty((n_land, N_MONTHS, len(CHANNELS)), dtype="float32")
tasks = [(ci, t, f) for ci, c in enumerate(CHANNELS) for t, f in enumerate(file_lists[c])]
def _fill(task):
    ci, t, f = task
    with rasterio.open(f) as s:
        a = s.read(1); nd = s.nodata
    v = a[LAND_ROWS, LAND_COLS].astype("float32")
    if nd is not None: v[v == nd] = np.nan
    X[:, t, ci] = v
t0 = time.time()
with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
    list(ex.map(_fill, tasks))
valid = np.isfinite(X).all(axis=(1, 2))
X = np.ascontiguousarray(X[valid])
VALID_ROWS, VALID_COLS = LAND_ROWS[valid], LAND_COLS[valid]
N = X.shape[0]
log(f"read {time.time()-t0:.1f}s  X: {X.shape} ({X.nbytes/1e9:.2f} GB)  dropped {n_land-N} NaN px")
assert N == 578_888, f"유효 셀 {N} != 578,888"

In [ ]:
# === Cell 3. 전처리 + Dataset + 헬퍼 + 학습/추출 루틴 =================
ch_mean = X.reshape(-1, X.shape[-1]).mean(0).astype("float32")
ch_std = (X.reshape(-1, X.shape[-1]).std(0) + 1e-6).astype("float32")
MEAN = torch.tensor(ch_mean, device=DEVICE); STD = torch.tensor(ch_std, device=DEVICE)
X_t = torch.from_numpy(X)

class SeqDS(torch.utils.data.Dataset):
    def __init__(self, t, idx=None): self.t = t; self.idx = idx
    def __len__(self): return self.t.shape[0] if self.idx is None else len(self.idx)
    def __getitem__(self, i): return self.t[i] if self.idx is None else self.t[self.idx[i]]

rng = np.random.default_rng(SEED)
_perm = rng.permutation(N)
val_idx = _perm[:VAL_SUBSAMPLE]
TRAIN_PX = (N - VAL_SUBSAMPLE) if TRAIN_SUBSAMPLE == "all" else int(TRAIN_SUBSAMPLE)
assert TRAIN_PX + VAL_SUBSAMPLE <= N
train_idx = _perm[VAL_SUBSAMPLE:VAL_SUBSAMPLE + TRAIN_PX]
assert len(np.intersect1d(train_idx, val_idx)) == 0
PIN = (DEVICE == "cuda")
train_loader = torch.utils.data.DataLoader(SeqDS(X_t, train_idx), batch_size=BATCH, shuffle=True, pin_memory=PIN)
val_loader = torch.utils.data.DataLoader(SeqDS(X_t, val_idx), batch_size=INFER_BATCH, shuffle=False, pin_memory=PIN)
eval_loader = torch.utils.data.DataLoader(SeqDS(X_t), batch_size=INFER_BATCH, shuffle=False, pin_memory=PIN)
log(f"train px={len(train_idx)} val px={len(val_idx)} eval px={N}")

def norm(xb): return (xb - MEAN) / STD

def span_mask(B, L, ratio, span, device):
    assert L % span == 0
    nb = L // span
    bm = torch.rand(B, nb, device=device) < ratio
    bm[bm.all(1), 0] = False
    return bm.repeat_interleave(span, dim=1)

with open(DEM_PATH) as _f: ASC_HEADER = [next(_f) for _ in range(6)]
def _scatter(v):
    g = np.full((G_H, G_W), -9999.0, dtype="float32"); g[VALID_ROWS, VALID_COLS] = v; return g
def write_asc(path, grid2d):
    with open(path, "w") as f: f.writelines(ASC_HEADER); np.savetxt(f, grid2d, fmt="%.6g")
def write_embedding(emb, algo, k, suffix=""):
    if algo == "rawPCA":
        d = os.path.join(OUTDIR, "rawPCA" + suffix); fmt = "rawPCA_PC{:02d}.asc"
    else:
        d = os.path.join(OUTDIR, f"CEV_{algo}{suffix}"); fmt = algo + "_CEV{:02d}.asc"
    os.makedirs(d, exist_ok=True)
    for b in range(k): write_asc(os.path.join(d, fmt.format(b + 1)), _scatter(emb[:, b]))
    log(f"saved {k} .asc -> {d}")

def save_cum(tag, cum):
    with open(os.path.join(OUTDIR, f"pca_cum_{tag}.csv"), "w", encoding="utf-8") as f:
        f.write("k,cum_var\n"); f.writelines(f"{i+1},{c:.6f}\n" for i, c in enumerate(cum))

def reduce_to_emb(reps, thresh=VAR_THRESH, max_k=MAX_K, tag=None):
    reps = (reps - reps.mean(0)) / (reps.std(0) + 1e-6)
    nc = min(max_k, reps.shape[1], reps.shape[0])
    p = PCA(n_components=nc, svd_solver="randomized", random_state=SEED).fit(reps)
    cum = np.cumsum(p.explained_variance_ratio_)
    assert cum[-1] >= thresh, f"MAX_K={max_k}로 누적분산 {thresh*100:.0f}% 미달 (최대 {cum[-1]*100:.1f}%)"
    k = int(np.searchsorted(cum, thresh) + 1)
    out = p.transform(reps)[:, :max(k, FIXED_K_SENS)]
    s = np.sign(np.sum(out ** 3, axis=0)); s[s == 0] = 1
    out *= s
    if tag is not None: save_cum(tag, cum)
    sens = "  ".join(f"{t:.2f}->k={int(np.searchsorted(cum, t) + 1)}" if cum[-1] >= t
                     else f"{t:.2f}->미달({nc}축 최대 {cum[-1]*100:.1f}%)" for t in VAR_THRESH_REPORT)
    log(f"  [{tag}] k={k} @ {cum[k-1]*100:.1f}%  | 임계값 민감도: {sens}")
    return out.astype("float32"), k

write_asc(os.path.join(OUTDIR, "cev_valid_mask.asc"), _scatter(np.ones(N, "float32")))

@torch.no_grad()
def val_mse(model, seed=VAL_SEED):
    was_train = model.training; model.eval()
    cpu_state = torch.random.get_rng_state()
    cuda_state = torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None
    torch.manual_seed(seed)
    tot = 0.0; n = 0
    for xb in val_loader:
        xb = norm(xb.to(DEVICE, non_blocking=True))
        tot += model(xb).item() * xb.shape[0]; n += xb.shape[0]
    torch.random.set_rng_state(cpu_state)
    if cuda_state is not None: torch.cuda.set_rng_state_all(cuda_state)
    if was_train: model.train()
    return tot / n

def train_model(model, tag):
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP); model.train()
    hist = []; best_va = float("inf"); best_state = None; t_start = time.time(); start_ep = 0
    ck = os.path.join(OUTDIR, f"ckpt_{tag}.pt")
    if RESUME and os.path.exists(ck):
        c = torch.load(ck, map_location="cpu")
        if c["n_heads"] != N_HEADS:
            sys.exit(f"RESUME: ckpt n_heads={c['n_heads']}, 현재 N_HEADS {N_HEADS}. 같은 값으로 실행할 것")
        model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"]); scaler.load_state_dict(c["scaler"])
        hist, best_va, best_state, start_ep = c["hist"], c["best_va"], c["best_state"], c["epoch"]
        torch.random.set_rng_state(c["rng_cpu"])
        if c["rng_cuda"] is not None: torch.cuda.set_rng_state_all(c["rng_cuda"])
        log(f"[{tag}] ckpt에서 이어서 학습: ep {start_ep} 완료 상태, 최저 val {best_va:.4f}")
    for ep in range(start_ep, EPOCHS):
        tot = 0.0; t0 = time.time()
        for xb in train_loader:
            xb = norm(xb.to(DEVICE, non_blocking=True)); opt.zero_grad(set_to_none=True)
            if USE_AMP:
                with torch.autocast("cuda"): loss = model(xb)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss = model(xb); loss.backward(); opt.step()
            tot += loss.item() * xb.shape[0]
        tr = tot / len(train_loader.dataset); va = val_mse(model)
        hist.append((ep + 1, tr, va, time.time() - t0))
        if va < best_va:
            best_va = va
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        if (ep + 1) % 5 == 0 or ep == 0:
            log(f"[{tag}] ep {ep+1:03d}/{EPOCHS} train {tr:.4f} | val {va:.4f} | {time.time()-t0:.1f}s  (누적 {(time.time()-t_start)/60:.0f}분)")
        torch.save({"epoch": ep + 1, "n_heads": N_HEADS, "model": model.state_dict(), "opt": opt.state_dict(),
                    "scaler": scaler.state_dict(), "hist": hist, "best_va": best_va, "best_state": best_state,
                    "rng_cpu": torch.random.get_rng_state(),
                    "rng_cuda": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}, ck + ".tmp")
        os.replace(ck + ".tmp", ck)
    best = min(hist, key=lambda r: r[2])
    model.load_state_dict(best_state)
    log(f"[{tag}] 최저 val MSE {best[2]:.4f} @ ep {best[0]} 가중치로 복원 (최종 ep{EPOCHS}: val {hist[-1][2]:.4f}, train {hist[-1][1]:.4f})")
    with open(os.path.join(OUTDIR, f"train_hist_{tag}.csv"), "w", encoding="utf-8") as f:
        f.write("epoch,train_mse,val_mse,sec\n"); f.writelines(f"{e},{a:.6f},{b:.6f},{s:.1f}\n" for e, a, b, s in hist)
    torch.save(best_state, os.path.join(OUTDIR, f"model_{tag}.pt"))
    if os.path.exists(ck): os.remove(ck)
    return best

def load_trained(model, tag):
    model.load_state_dict(torch.load(os.path.join(OUTDIR, f"model_{tag}.pt"), map_location=DEVICE))
    hist = np.genfromtxt(os.path.join(OUTDIR, f"train_hist_{tag}.csv"), delimiter=",", names=True)
    i = int(np.argmin(hist["val_mse"]))
    best = (int(hist["epoch"][i]), float(hist["train_mse"][i]), float(hist["val_mse"][i]), float(hist["sec"][i]))
    log(f"[{tag}] 저장 가중치 불러옴(학습 생략) — 최저 val MSE {best[2]:.4f} @ ep {best[0]} / 이력 {len(hist)} epoch")
    return best

@torch.no_grad()
def extract(model, feat_dim):
    model.eval(); reps = np.empty((N, feat_dim), dtype="float32"); pos = 0
    for xb in eval_loader:
        e = model.embed(norm(xb.to(DEVICE, non_blocking=True))).float().cpu().numpy()
        reps[pos:pos + e.shape[0]] = e; pos += e.shape[0]
    return reps

info = {"seed": SEED, "train_px": int(len(train_idx)), "val_px": VAL_SUBSAMPLE, "epochs": EPOCHS, "batch": BATCH,
        "device": GPU_NAME, "amp": USE_AMP, "d_model": D_MODEL, "n_heads": N_HEADS, "e_layers": E_LAYERS, "d_ff": D_FF,
        "dropout": DROPOUT, "mask_ratio": MASK_RATIO, "patch": PATCH_LEN, "stride": STRIDE, "factor": INFORMER_FACTOR,
        "lr": LR, "var_thresh": VAR_THRESH, "max_k": MAX_K, "k": {}, "best_epoch": {}, "best_val_mse": {}, "minutes": {}}
models = [m.strip().lower() for m in MODELS.split(",")]
_ri = os.path.join(OUTDIR, "run_info.json")
if (LOAD_WEIGHTS or RESUME) and os.path.exists(_ri):
    _h = json.load(open(_ri, encoding="utf-8")).get("n_heads")
    if _h is not None and _h != N_HEADS:
        sys.exit(f"LOAD_WEIGHTS: 저장 가중치 n_heads={_h}, 현재 N_HEADS {N_HEADS}. 같은 값으로 실행할 것")

## PatchTST

In [ ]:
# === Cell 4. PatchTST 정의 ============================================
class PatchTST(nn.Module):
    def __init__(self, seq_len, n_ch, patch_len, stride, d_model, n_heads, e_layers, d_ff, dropout):
        super().__init__()
        assert stride >= patch_len, "패치 중첩 금지: 마스킹 구간이 이웃 패치에 노출된다"
        self.patch_len, self.stride, self.n_ch = patch_len, stride, n_ch
        self.n_patch = (seq_len - patch_len) // stride + 1
        self.proj_in = nn.Linear(patch_len, d_model)
        self.pos = nn.Parameter(torch.randn(1, self.n_patch, d_model) * 0.02)
        self.mask_tok = nn.Parameter(torch.randn(d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc, e_layers)
        self.recon = nn.Linear(d_model, patch_len); self.feat_dim = n_ch * d_model
    def patchify(self, x):
        x = x.permute(0, 2, 1).unfold(-1, self.patch_len, self.stride)
        return x.reshape(x.shape[0] * self.n_ch, self.n_patch, self.patch_len)
    def encode(self, x, mask=None):
        p = self.patchify(x); tok = self.proj_in(p)
        if mask is not None: tok = torch.where(mask.unsqueeze(-1), self.mask_tok, tok)
        return self.encoder(tok + self.pos), p
    def forward(self, x):
        p = self.patchify(x); BC, NP, _ = p.shape
        mask = torch.rand(BC, NP, device=x.device) < MASK_RATIO
        mask[mask.all(1), 0] = False
        h, target = self.encode(x, mask); pred = self.recon(h)
        return ((pred - target) ** 2 * mask.unsqueeze(-1)).sum() / (mask.sum() * self.patch_len + 1e-6)
    @torch.no_grad()
    def embed(self, x):
        h, _ = self.encode(x, None); h = h.mean(1)
        return h.reshape(x.shape[0], self.n_ch, -1).reshape(x.shape[0], -1)

In [ ]:
# === Cell 5. PatchTST 학습 + 임베딩 추출 + 전세계 .asc 저장 ===========
if "patchtst" in models:
    t0 = time.time(); torch.manual_seed(SEED)
    m = PatchTST(N_MONTHS, len(CHANNELS), PATCH_LEN, STRIDE, D_MODEL, N_HEADS, E_LAYERS, D_FF, DROPOUT).to(DEVICE)
    log(f"[PatchTST] n_patch={m.n_patch}  params={sum(p.numel() for p in m.parameters())}")
    loaded = (LOAD_WEIGHTS or RESUME) and os.path.exists(os.path.join(OUTDIR, "model_PatchTST.pt"))
    best = load_trained(m, "PatchTST") if loaded else train_model(m, "PatchTST")
    reps = extract(m, m.feat_dim)
    if SAVE_REPS: np.save(os.path.join(OUTDIR, "reps_PatchTST.npy"), reps)
    emb, k = reduce_to_emb(reps, tag="PatchTST"); write_embedding(emb[:, :k], "PatchTST", k)
    write_embedding(emb[:, :FIXED_K_SENS], "PatchTST", FIXED_K_SENS, "_k19")
    _, k_cm = reduce_to_emb(reps.reshape(N, len(CHANNELS), D_MODEL).mean(1), tag="PatchTST_chmean")
    log(f"[진단] PatchTST 채널concat(384) k={k} vs 채널평균(128) k={k_cm}")
    info["k"]["PatchTST"] = k; info["k"]["PatchTST_chmean"] = k_cm
    info["best_epoch"]["PatchTST"] = best[0]; info["best_val_mse"]["PatchTST"] = best[2]; info["minutes"]["PatchTST"] = (time.time() - t0) / 60
    del m, reps, emb; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    json.dump(info, open(os.path.join(OUTDIR, "run_info.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## Informer

In [ ]:
# === Cell 6. Informer 정의 (ProbSparse attention) =====================
class ProbAttention(nn.Module):
    def __init__(self, factor=5, dropout=0.1):
        super().__init__(); self.factor = factor; self.drop = nn.Dropout(dropout)
    def _prob_QK(self, Q, K, sample_k, n_top):
        B, H, L_K, E = K.shape; _, _, L_Q, _ = Q.shape
        K_exp = K.unsqueeze(-3).expand(B, H, L_Q, L_K, E)
        idx = torch.randint(L_K, (L_Q, sample_k), device=K.device)
        K_smp = K_exp[:, :, torch.arange(L_Q).unsqueeze(1), idx, :]
        Q_K_s = torch.matmul(Q.unsqueeze(-2), K_smp.transpose(-2, -1)).squeeze(-2)
        M = Q_K_s.max(-1)[0] - Q_K_s.sum(-1) / L_K; M_top = M.topk(n_top, sorted=False)[1]
        Q_red = Q[torch.arange(B)[:, None, None], torch.arange(H)[None, :, None], M_top, :]
        return torch.matmul(Q_red, K.transpose(-2, -1)), M_top
    def _context(self, V, L_Q):
        return V.mean(-2).unsqueeze(-2).expand(*V.shape[:2], L_Q, V.shape[-1]).clone()
    def _update(self, ctx, V, scores, idx, L_Q):
        B, H = V.shape[0], V.shape[1]; attn = torch.softmax(scores, -1)
        ctx[torch.arange(B)[:, None, None], torch.arange(H)[None, :, None], idx, :] = \
            torch.matmul(self.drop(attn), V).type_as(ctx)
        return ctx
    def forward(self, q, k, v):
        B, L_Q, H, E = q.shape; L_K = k.shape[1]
        q, k, v = q.transpose(2, 1), k.transpose(2, 1), v.transpose(2, 1)
        U = min(self.factor * int(np.ceil(np.log(L_K))), L_K); u = min(self.factor * int(np.ceil(np.log(L_Q))), L_Q)
        scores, idx = self._prob_QK(q, k, U, u); scores = scores / math.sqrt(E)
        ctx = self._update(self._context(v, L_Q), v, scores, idx, L_Q)
        return ctx.transpose(2, 1).contiguous()
class AttentionLayer(nn.Module):
    def __init__(self, d_model, n_heads, factor, dropout):
        super().__init__(); self.dk = d_model // n_heads; self.h = n_heads
        self.q = nn.Linear(d_model, d_model); self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model); self.o = nn.Linear(d_model, d_model)
        self.att = ProbAttention(factor=factor, dropout=dropout)
    def forward(self, x):
        B, L, _ = x.shape
        q = self.q(x).view(B, L, self.h, self.dk); k = self.k(x).view(B, L, self.h, self.dk); v = self.v(x).view(B, L, self.h, self.dk)
        return self.o(self.att(q, k, v).view(B, L, -1))
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, factor, dropout):
        super().__init__(); self.att = AttentionLayer(d_model, n_heads, factor, dropout)
        self.n1 = nn.LayerNorm(d_model); self.n2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.n1(x + self.drop(self.att(x))); return self.n2(x + self.drop(self.ff(x)))
class Informer(nn.Module):
    def __init__(self, seq_len, n_ch, d_model, n_heads, e_layers, d_ff, factor, dropout):
        super().__init__(); self.seq_len = seq_len; self.n_ch = n_ch; self.feat_dim = d_model
        self.val_emb = nn.Conv1d(n_ch, d_model, 3, padding=1)
        self.pos = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        self.mask_tok = nn.Parameter(torch.randn(d_model) * 0.02)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, factor, dropout) for _ in range(e_layers)])
        self.recon = nn.Linear(d_model, n_ch)
    def encode(self, x, mask=None):
        if mask is not None: x = x.masked_fill(mask.unsqueeze(-1), 0.0)
        h = self.val_emb(x.permute(0, 2, 1)).permute(0, 2, 1)
        if mask is not None: h = torch.where(mask.unsqueeze(-1), self.mask_tok, h)
        h = h + self.pos
        for lyr in self.layers: h = lyr(h)
        return h
    def forward(self, x):
        B, L, _ = x.shape
        mask = span_mask(B, L, MASK_RATIO, MASK_SPAN, x.device)
        pred = self.recon(self.encode(x, mask))
        return ((pred - x) ** 2 * mask.unsqueeze(-1)).sum() / (mask.sum() * self.n_ch + 1e-6)
    @torch.no_grad()
    def embed(self, x): return self.encode(x, None).mean(1)

In [ ]:
# === Cell 7. Informer 학습 + 임베딩 추출 + 전세계 .asc 저장 ===========
if "informer" in models:
    t0 = time.time(); torch.manual_seed(SEED)
    m = Informer(N_MONTHS, len(CHANNELS), D_MODEL, N_HEADS, E_LAYERS, D_FF, INFORMER_FACTOR, DROPOUT).to(DEVICE)
    log(f"[Informer] params={sum(p.numel() for p in m.parameters())}")
    loaded = (LOAD_WEIGHTS or RESUME) and os.path.exists(os.path.join(OUTDIR, "model_Informer.pt"))
    best = load_trained(m, "Informer") if loaded else train_model(m, "Informer")
    reps = extract(m, m.feat_dim)
    if SAVE_REPS: np.save(os.path.join(OUTDIR, "reps_Informer.npy"), reps)
    emb, k = reduce_to_emb(reps, tag="Informer"); write_embedding(emb[:, :k], "Informer", k)
    write_embedding(emb[:, :FIXED_K_SENS], "Informer", FIXED_K_SENS, "_k19")
    info["k"]["Informer"] = k
    info["best_epoch"]["Informer"] = best[0]; info["best_val_mse"]["Informer"] = best[2]; info["minutes"]["Informer"] = (time.time() - t0) / 60
    del m, reps, emb; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    json.dump(info, open(os.path.join(OUTDIR, "run_info.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)

## RawPCA

In [ ]:
# === Cell 8. RawPCA + 전세계 .asc 저장 ================================
if "rawpca" in models:
    t0 = time.time()
    rng_raw = np.random.default_rng(SEED + 1)
    fit_idx = rng_raw.choice(N, min(RAW_FIT_N, N), replace=False)
    FLAT_D = X.shape[1] * X.shape[2]
    _fit = X[fit_idx].reshape(len(fit_idx), FLAT_D)
    raw_mu = _fit.mean(0); raw_sd = _fit.std(0) + 1e-6
    nc_raw = min(MAX_K, FLAT_D)
    p_raw = PCA(n_components=nc_raw, svd_solver="randomized", random_state=SEED).fit((_fit - raw_mu) / raw_sd)
    del _fit
    cum_raw = np.cumsum(p_raw.explained_variance_ratio_)
    assert cum_raw[-1] >= VAR_THRESH, f"raw-PCA: MAX_K={MAX_K}로 미달 (최대 {cum_raw[-1]*100:.1f}%)"
    k_raw = int(np.searchsorted(cum_raw, VAR_THRESH) + 1); save_cum("rawPCA", cum_raw)
    log(f"[rawPCA] k={k_raw} @ {cum_raw[k_raw-1]*100:.1f}%")
    kk_raw = max(k_raw, FIXED_K_SENS)
    emb_raw = np.empty((N, kk_raw), dtype="float32")
    for s in range(0, N, 20_000):
        blk = (X[s:s + 20_000].reshape(-1, FLAT_D) - raw_mu) / raw_sd
        emb_raw[s:s + 20_000] = p_raw.transform(blk)[:, :kk_raw]
    sgn = np.sign(np.sum(emb_raw.astype("float64") ** 3, axis=0)); sgn[sgn == 0] = 1
    emb_raw *= sgn.astype("float32")
    write_embedding(emb_raw[:, :k_raw], "rawPCA", k_raw)
    write_embedding(emb_raw[:, :FIXED_K_SENS], "rawPCA", FIXED_K_SENS, "_k19")
    info["k"]["rawPCA"] = k_raw; info["minutes"]["rawPCA"] = (time.time() - t0) / 60
    json.dump(info, open(os.path.join(OUTDIR, "run_info.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)

log(f"=== 완료 seed={SEED}: {json.dumps(info['k'])}  minutes={ {k: round(v, 1) for k, v in info['minutes'].items()} } ===")
LOG.close()